# Causal-GPT-RL Hugging Face Hub Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccnets-team/causal-gpt-rl/blob/main/examples/hub_quickstart.ipynb)

This first example loads an exported policy bundle from Hugging Face Hub, creates the matching Gymnasium environment, runs a small no-render evaluation, and prints a compact result summary.

The default bundle layout follows the README convention: one Hub repo with one subfolder per environment.

It runs on Colab unchanged. A GPU runtime is optional: one environment steps one
row at a time, which is small enough for CPU.

## Install

Run this once in a fresh notebook runtime. `mujoco` is pinned to `3.2.3` because
that is the version the Minari datasets were recorded with, and a different
simulator release is a different measurement even with identical weights and
seeds.

In [ ]:
%pip install -q "causal-gpt-rl[hub,mujoco]" "mujoco==3.2.3"

## Imports

In [ ]:
import gymnasium as gym
import torch

from causal_gpt_rl.inference import load_runner_from_hub, run_episodes

### Runtime check

The published numbers were measured on a recorded stack. Colab's preinstalled
`torch` will usually differ, which is worth seeing rather than discovering later
from a return that does not match.

In [ ]:
import mujoco

PROTOCOL_STACK = {"torch": "2.8.0", "gymnasium": "1.2.3", "mujoco": "3.2.3"}
installed = {
    "torch": torch.__version__,
    "gymnasium": gym.__version__,
    "mujoco": mujoco.__version__,
}

for name, reference in PROTOCOL_STACK.items():
    # A local version segment ("2.8.0+cu129") is a build of the same release.
    agrees = installed[name].split("+")[0] == reference
    print(f"{'  ' if agrees else '* '}{name:<11}{installed[name]:<18}card: {reference}")

## Choose a Hub Bundle

The example below loads this path:

```text
ccnets/causal-gpt-rl/
  ant-v5/
    config.json
    model.safetensors
```

In [ ]:
repo_id = "ccnets/causal-gpt-rl"
subfolder = "ant-v5"
env_id = "Ant-v5"

num_episodes = 5
seed = 0
max_steps = None
device = "cuda" if torch.cuda.is_available() else "cpu"

# For a private Hub repo, log in first and set this to True.
hub_token = None

config = {
    "repo_id": repo_id,
    "subfolder": subfolder,
    "env_id": env_id,
    "num_episodes": num_episodes,
    "seed": seed,
    "device": device,
}

config

## Load the Policy Runner

In [ ]:
env = gym.make(env_id)
runner = load_runner_from_hub(
    repo_id,
    subfolder=subfolder,
    token=hub_token,
    device=device,
)

print(f"Loaded {repo_id}/{subfolder} for {env_id} on {device}.")
runner

## Run a Small Evaluation

This keeps the first run short. Increase `num_episodes` for a more stable score.

In [ ]:
try:
    stats = run_episodes(
        env,
        runner,
        num_episodes=num_episodes,
        seed=seed,
        max_steps=max_steps,
    )
finally:
    env.close()

stats

## Summary

In [ ]:
summary = {
    "bundle": f"{repo_id}/{subfolder}",
    "env_id": env_id,
    "device": device,
    "return_mean": round(float(stats["return_mean"]), 3),
    "return_std": round(float(stats["return_std"]), 3),
    "length_mean": round(float(stats["length_mean"]), 3),
    "length_std": round(float(stats["length_std"]), 3),
    "num_episodes": int(stats["num_episodes"]),
}

for key, value in summary.items():
    print(f"{key:>12}: {value}")

summary

## This is a smoke test, not the protocol

Five episodes off one seed land wherever the draw put them. The published scores
are 50 episodes with seeds `0..49`, run together as one 50-row batch, and
`run_episodes` seeds only its first reset — it cannot express that seed range.

Measuring a bundle the way its model card reports it takes
[`examples/deploy/reproduce.py`](https://github.com/ccnets-team/causal-gpt-rl/blob/main/examples/deploy/reproduce.py),
which ships in the repository rather than in the installed package. These are
notebook magics, not shell commands — paste them into a new cell:

```
!git clone -q https://github.com/ccnets-team/causal-gpt-rl.git
%cd causal-gpt-rl
!python -m examples.deploy.reproduce --env-id Ant-v5
```